# ShelfSense — Exploratory Data Analysis

Explores the price warehouse through the analytical views in `database/views/analytical_views.sql`.

**Prerequisites:** run `python scripts/setup_database.py --with-data` (or the daily pipeline) so the tables and views are populated.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

from config.db_config import CONNECTION_STRING

engine = create_engine(CONNECTION_STRING)
plt.rcParams.update({"figure.figsize": (11, 5), "axes.spines.top": False, "axes.spines.right": False})

RETAILER_COLORS = {"BigBasket": "#84c225", "Blinkit": "#f8cb46", "Zepto": "#5a1e96", "JioMart": "#0078ad"}

def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

## 1. Dataset overview

In [ ]:
q("""
    SELECT
        (SELECT COUNT(*) FROM dim_product)                        AS products,
        (SELECT COUNT(*) FROM dim_retailer)                       AS retailers,
        (SELECT COUNT(*) FROM fact_price_history)                 AS price_observations,
        (SELECT MIN(price_date) FROM fact_price_history)          AS first_date,
        (SELECT MAX(price_date) FROM fact_price_history)          AS last_date,
        (SELECT ROUND(AVG((NOT is_available)::INT) * 100, 2)
           FROM fact_price_history)                               AS stockout_rate_pct
""")

## 2. Who discounts most aggressively? (last 7 days)

In [ ]:
weekly = q("SELECT * FROM vw_weekly_discount_analysis").set_index("category_name")
retailer_cols = ["bigbasket_avg_discount", "blinkit_avg_discount", "zepto_avg_discount", "jiomart_avg_discount"]
weekly[retailer_cols].mean().rename(lambda c: c.split("_")[0].title()).sort_values().plot.barh(color="#4c72b0")
plt.xlabel("Average discount (%)")
plt.title("Average discount by retailer — last 7 days")
plt.show()
weekly.head(10)

## 3. Weekend vs weekday promotion strategy

In [ ]:
wk = q("SELECT * FROM vw_weekend_vs_weekday_discounts")
lift = wk.groupby("retailer_name")["weekend_discount_lift_ppt"].mean().sort_values()
lift.plot.barh(color=[RETAILER_COLORS[r] for r in lift.index])
plt.xlabel("Weekend discount lift (percentage points)")
plt.title("How much deeper do retailers discount on weekends?")
plt.show()
pd.crosstab(wk["retailer_name"], wk["pricing_strategy"])

## 4. Who wins on price most often?

In [ ]:
cheapest = q("""
    SELECT retailer_name, SUM(days_cheapest) AS days_cheapest
    FROM vw_cheapest_retailer_frequency
    GROUP BY retailer_name
""").set_index("retailer_name")["days_cheapest"]
share = (cheapest / cheapest.sum() * 100).sort_values()
share.plot.barh(color=[RETAILER_COLORS[r] for r in share.index])
plt.xlabel("Share of product-days cheapest (%)")
plt.title("Cheapest-retailer frequency")
plt.show()

## 5. Category price index (day 1 = 100, 7-day moving average)

In [ ]:
idx = q("SELECT price_date, parent_category, price_index_7day_ma FROM vw_category_price_index")
by_parent = idx.groupby(["price_date", "parent_category"])["price_index_7day_ma"].mean().unstack()
by_parent.plot()
plt.axhline(100, color="grey", lw=0.8, ls="--")
plt.ylabel("Price index")
plt.title("Price index by parent category")
plt.show()

## 6. Price volatility — which products need closest monitoring?

In [ ]:
vol = q("SELECT * FROM vw_price_volatility")
vol.groupby("category_name")["volatility_index_pct"].mean().sort_values().plot.barh(color="#dd8452")
plt.xlabel("Coefficient of variation (%)")
plt.title("Average 30-day price volatility by category")
plt.show()
vol.head(10)

## 7. Inflation: absorbing or passing on?

In [ ]:
infl = q("SELECT * FROM vw_inflation_adjusted_pricing WHERE cpi_index IS NOT NULL")
pd.crosstab(infl["category_name"], infl["inflation_strategy"])

## 8. Promotional calendar — longest promotions

In [ ]:
q("""
    SELECT retailer_name, product_name, promo_start_date, promo_end_date,
           promo_duration_days, avg_promo_discount
    FROM vw_promotional_calendar
    ORDER BY promo_duration_days DESC
    LIMIT 15
""")